# Phase 3 — ML feature importance

Train logistic regression or XGBoost on **in-sample** data and plot which features drive long/cash signals.

Run from project root with the venv active:
```bash
jupyter notebook notebooks/phase3_ml.ipynb
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import IN_SAMPLE_FRAC, split_in_out_sample
from src.data_loader import UNIVERSE, load_prices
from src.features import WARMUP_BARS, build_dataset
from src.ml_strategy import DEFAULT_PROB_THRESHOLD, feature_importance, train_model

TICKER = "VOO"
MODEL_TYPE = "logreg"  # or "xgb"
THRESHOLD = DEFAULT_PROB_THRESHOLD

In [ ]:
full = load_prices(TICKER)
in_df, out_df, in_period, out_period = split_in_out_sample(
    full, IN_SAMPLE_FRAC, min_warmup=WARMUP_BARS
)
print(f"In-sample:  {in_period.start.date()} → {in_period.end.date()}")
print(f"Out-sample: {out_period.start.date()} → {out_period.end.date()}")

dataset = build_dataset(in_df)
print(f"\nTraining rows: {len(dataset)}")
print(f"Label balance (% long): {dataset['label'].mean():.1%}")
dataset.head()

In [ ]:
trained = train_model(
    in_df,
    TICKER,
    prob_threshold=THRESHOLD,
    model_type=MODEL_TYPE,
)
imp = feature_importance(trained)
imp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
imp.sort_values().plot(kind="barh", ax=ax, color="steelblue")
title = "Logistic regression |coef|" if MODEL_TYPE == "logreg" else "XGBoost gain importance"
ax.set_title(f"{TICKER} — {title}")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Compare both models side by side
rows = []
for model_type in ("logreg", "xgb"):
    m = train_model(in_df, TICKER, prob_threshold=THRESHOLD, model_type=model_type)
    rows.append(feature_importance(m).rename(model_type))
pd.concat(rows, axis=1)